In [ ]:
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState, StateGraph,END
import os
from logger import Colors, log_error, log_header, log_info, log_success, log_warning
from langgraph.prebuilt import ToolNode


load_dotenv(override=True) 

True

In [2]:
nvidia_api_key = os.getenv("NVIDIA_API_KEY")

if nvidia_api_key is None:
    log_warning("NVIDIA API Key not found in environment variables.")
else:
    log_info(f"NVIDIA API Key loaded successfully: {nvidia_api_key[:4]}...")


ℹ️  NVIDIA API Key loaded successfully: nvap...


In [3]:
@tool
def triple(num:float) -> float:
    """Triples the input number."""
    log_info(f"Tripling the number: {num}")
    return num * 3

In [4]:
tools = [triple, TavilySearch(max_results=2)]
MODEL = ChatNVIDIA(model="moonshotai/kimi-k2-instruct-0905").bind_tools(tools)

In [5]:
SYSTEM_MESSAGE = """You are a helpful assistant that can perform various tasks using the tools provided."""

In [6]:
def run_agent_reasoning(state: MessagesState) -> MessagesState:
    """Run the agent reasoning node"""
    log_info("Running agent reasoning...")
    response = MODEL.invoke([{"role": "system", "content": SYSTEM_MESSAGE} ,*state["messages"]])
    log_info(f"Agent reasoning response: {response}")
    return {"messages": [response]}

In [7]:
tool_node = ToolNode(tools=tools)

In [8]:
AGENT_REASON="agent_reason"
ACT= "act"
LAST = -1

In [9]:

def should_continue(state: MessagesState) -> str:
    if not state["messages"][LAST].tool_calls:
        log_info("No tool calls detected, ending the agent.")
        return END
    log_info("Tool calls detected, continuing the agent.")
    return ACT

In [10]:
flow = StateGraph(MessagesState)

flow.add_node(AGENT_REASON, run_agent_reasoning)
flow.set_entry_point(AGENT_REASON)
flow.add_node(ACT, tool_node)
flow.add_conditional_edges(AGENT_REASON, should_continue, {
    END:END,
    ACT:ACT})
flow.add_edge(ACT, AGENT_REASON)

app = flow.compile()
# app.get_graph().draw_mermaid_png(output_file_path="flow.png")


In [11]:
print("Hello ReAct LangGraph with Function Calling")
res = app.invoke({"messages": [HumanMessage(content="What is the temperature in Tokyo? List it and then triple it")]})
print(res["messages"][LAST].content)

Hello ReAct LangGraph with Function Calling
ℹ️  Running agent reasoning...
ℹ️  Agent reasoning response: content='' additional_kwargs={'tool_calls': [{'id': 'functions.tavily_search:0', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current temperature in Tokyo"}'}}]} response_metadata={'role': 'assistant', 'content': None, 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'functions.tavily_search:0', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current temperature in Tokyo"}'}}], 'reasoning': None, 'reasoning_content': None, 'token_usage': {'prompt_tokens': 1650, 'total_tokens': 1744, 'completion_tokens': 94, 'prompt_tokens_details': None}, 'finish_reason': 'tool_calls', 'model_name': 'moonshotai/kimi-k2-instruct-0905'} id='lc_run--019dc9ad-1f62-7370-a5e7-28ac8812f9b3-0' tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current temperature in Tokyo'}, 'id': 